# Reverse PCA: fit on AoU, project 1000G onto it

Every other notebook fits the PCA on 1000G and projects AoU onto it. This one runs it backwards, as an independent cross-check: fit a PCA on a random subsample of AoU itself (`N_AOU_PCA_SAMPLES`, a couple thousand -- plenty for a stable PCA, far cheaper than all 400k+), project the 1000G EUR reference samples onto *that* AoU-defined space, and reproject the AoU subsample onto its own PCs as a sanity check. If AoU's own structure and the 1000G-onto-1000G picture from `round1_filter.ipynb`/`round2_filter.ipynb` broadly agree (1000G EUR populations cluster recognizably within AoU's own PC space, in roughly the same relative arrangement), that's independent support for the forward-direction results -- not just the same computation checked against itself.

LD pruning starts from `agreeing_snps.ids` (round 1's already ID+REF+ALT-verified, HM3-restricted variant set) rather than AoU's full variant catalog -- keeps the AoU-fit PCA and the reference projection on a variant set already known to cross-match cleanly, without needing a second harmonization pass.

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Inputs

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/01_ancestry_filtering"

# from round 1 (submit_pca_r1.ipynb)
OUT_PREFIX = f"{BUCKET_DIR}/1kg_all_hm3"                                    # QC'd, whole-cohort 1000G bfile
PANEL_PATH = f"{BUCKET_DIR}/integrated_call_samples_v3.20130502.ALL.panel"
ACAF_BIALLELIC_PREFIX = f"{BUCKET_DIR}/round1_pca/acaf_hm3_subset_biallelic"  # already biallelic-filtered, HM3-restricted ACAF
AGREEING_SNPS_PATH = f"{BUCKET_DIR}/round1_pca/agreeing_snps.ids"            # ID+REF+ALT-verified vs the reference

# from round 1 filtering (round1_filter.ipynb) -- prob_tag there is f"p{THRESHOLD_QUANTILE*100:g}",
# so THRESHOLD_QUANTILE=0.999 renders as "p99.9", not "p999" -- confirm this matches what that
# notebook's OUT_DIR + prob_tag actually produced before running
ROUND1_KEEP_PATH = f"{BUCKET_DIR}/round1_pca/round1_keep_ids_eur_p99.9.txt"

EUR_REF_POPS = ["CEU", "GBR", "TSI", "FIN", "IBS"]   # reference samples to project onto the AoU-fit PCs

N_AOU_PCA_SAMPLES = 2000
RANDOM_SEED = 0
N_PCS_TO_PLOT = 6
N_TOP_LOADINGS = 20

OUT_DIR = f"{BUCKET_DIR}/reverse_pca_aou"
os.makedirs(OUT_DIR, exist_ok=True)

AOU_SAMPLE_PATH = os.path.join(OUT_DIR, "aou_pca_sample.ids")
AOU_PRUNE_PREFIX = os.path.join(OUT_DIR, "aou_pruned")
AOU_PCA_PREFIX = os.path.join(OUT_DIR, "aou_pca")
REF_KEEP_PATH = os.path.join(OUT_DIR, "eur_ref_keep.ids")
REF_PROJECT_PREFIX = os.path.join(OUT_DIR, "ref_projected_onto_aou")
AOU_REPROJECT_KEEP_PATH = os.path.join(OUT_DIR, "aou_reproject_keep.ids")   # AOU_SAMPLE_PATH ∩ ROUND1_KEEP_PATH
AOU_REPROJECT_PREFIX = os.path.join(OUT_DIR, "aou_pca_reprojected")

print(OUT_DIR)

## Sample AoU individuals for the PCA fit

In [ ]:
import pandas as pd

psam = pd.read_csv(f"{ACAF_BIALLELIC_PREFIX}.psam", sep="\t")
psam_id_col = "#IID" if "#IID" in psam.columns else "IID"

aou_sample_ids = psam[psam_id_col].sample(n=N_AOU_PCA_SAMPLES, random_state=RANDOM_SEED)
aou_sample_ids.to_csv(AOU_SAMPLE_PATH, index=False, header=False)
print(f"Sampled {len(aou_sample_ids)} of {len(psam)} AoU individuals -> {AOU_SAMPLE_PATH}")

## LD prune + PCA, fit on the AoU subsample

`--nonfounders`: same reasoning as everywhere else in this pipeline -- ACAF's psam has no real pedigree, "founder" status is meaningless here. No HWE filter here (unlike round 2's within-population HWE step): this AoU subsample is a broad, not-yet-ancestry-restricted draw, analogous to round 1's whole-cohort 1000G PCA -- HWE filtering would fight against the very structure this PCA is meant to reveal.

In [ ]:
%%bash -s "$ACAF_BIALLELIC_PREFIX" "$AGREEING_SNPS_PATH" "$AOU_SAMPLE_PATH" "$AOU_PRUNE_PREFIX" "$AOU_PCA_PREFIX"
set -e
ACAF_BIALLELIC_PREFIX=$1
AGREEING_SNPS_PATH=$2
AOU_SAMPLE_PATH=$3
AOU_PRUNE_PREFIX=$4
AOU_PCA_PREFIX=$5

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$AOU_SAMPLE_PATH" \
  --extract "$AGREEING_SNPS_PATH" \
  --nonfounders \
  --indep-pairwise 200kb 1 0.1 \
  --out "$AOU_PRUNE_PREFIX"

wc -l "${AOU_PRUNE_PREFIX}.prune.in"

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$AOU_SAMPLE_PATH" \
  --extract "${AOU_PRUNE_PREFIX}.prune.in" \
  --nonfounders \
  --freq counts \
  --pca allele-wts 20 \
  --out "$AOU_PCA_PREFIX"

ls -lh "${AOU_PCA_PREFIX}".*

## Reference EUR keep-list

In [ ]:
panel = pd.read_csv(PANEL_PATH, sep="\t")
ref_keep_ids = panel.loc[panel["pop"].isin(EUR_REF_POPS), "sample"]
ref_keep_ids.to_csv(REF_KEEP_PATH, index=False, header=False)
print(f"{len(ref_keep_ids)} reference samples ({'+'.join(EUR_REF_POPS)}) -> {REF_KEEP_PATH}")

## Project the 1000G EUR reference onto the AoU-fit PCs

AVG scoring (no `sum` modifier) and `list-variants`, same conventions as every other projection in this pipeline -- both sides of the comparisons below are `--score` output scored identically, and `list-variants` records the exact variant set this run actually used for the reprojection sanity check to reuse.

In [ ]:
%%bash -s "$OUT_PREFIX" "$AOU_PCA_PREFIX" "$AOU_PRUNE_PREFIX" "$REF_KEEP_PATH" "$REF_PROJECT_PREFIX"
set -e
OUT_PREFIX=$1
AOU_PCA_PREFIX=$2
AOU_PRUNE_PREFIX=$3
REF_KEEP_PATH=$4
REF_PROJECT_PREFIX=$5

WEIGHTS="${AOU_PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

plink2 \
  --bfile "$OUT_PREFIX" \
  --keep "$REF_KEEP_PATH" \
  --extract "${AOU_PRUNE_PREFIX}.prune.in" \
  --nonfounders \
  --read-freq "${AOU_PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize list-variants \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$REF_PROJECT_PREFIX"

head "${REF_PROJECT_PREFIX}.sscore"
wc -l "${REF_PROJECT_PREFIX}.sscore.vars"

## Sanity check: reproject the round-1-passing AoU samples onto their own PCs

Same check used everywhere else in this pipeline -- should closely recover the direct `--pca` eigenvectors (up to arbitrary per-PC sign). Restricted to two intersections: `${REF_PROJECT_PREFIX}.sscore.vars` for variants (the `list-variants` output above, not a theoretical intersection -- the lesson from every earlier round of this debugging), and `AOU_SAMPLE_PATH ∩ ROUND1_KEEP_PATH` for samples -- only the AoU-fit subsample members that already passed round 1's EUR ellipsoid filter, not the full (ancestry-unrestricted) PCA-fit subsample.

In [ ]:
%%bash -s "$ACAF_BIALLELIC_PREFIX" "$AOU_REPROJECT_KEEP_PATH" "$AOU_PCA_PREFIX" "$REF_PROJECT_PREFIX" "$AOU_REPROJECT_PREFIX"
set -e
ACAF_BIALLELIC_PREFIX=$1
AOU_REPROJECT_KEEP_PATH=$2
AOU_PCA_PREFIX=$3
REF_PROJECT_PREFIX=$4
AOU_REPROJECT_PREFIX=$5

USED_VARS="${REF_PROJECT_PREFIX}.sscore.vars"
if [ ! -s "$USED_VARS" ]; then
  echo "Missing $USED_VARS -- run the reference projection cell (with 'list-variants') first" >&2
  exit 1
fi
if [ ! -s "$AOU_REPROJECT_KEEP_PATH" ]; then
  echo "Missing $AOU_REPROJECT_KEEP_PATH -- run the round-1-intersection cell above first" >&2
  exit 1
fi

WEIGHTS="${AOU_PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

plink2 \
  --pfile "$ACAF_BIALLELIC_PREFIX" \
  --keep "$AOU_REPROJECT_KEEP_PATH" \
  --extract "$USED_VARS" \
  --nonfounders \
  --read-freq "${AOU_PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$AOU_REPROJECT_PREFIX"

head "${AOU_REPROJECT_PREFIX}.sscore"

In [ ]:
aou_sample_ids_set = pd.read_csv(AOU_SAMPLE_PATH, header=None)[0]
round1_keep_ids_set = pd.read_csv(ROUND1_KEEP_PATH, header=None)[0]
aou_reproject_keep = aou_sample_ids_set[aou_sample_ids_set.isin(set(round1_keep_ids_set))]
aou_reproject_keep.to_csv(AOU_REPROJECT_KEEP_PATH, index=False, header=False)
print(f"{len(aou_reproject_keep)} of {len(aou_sample_ids_set)} AoU PCA-fit samples already passed round 1 -> {AOU_REPROJECT_KEEP_PATH}")

In [ ]:
direct = pd.read_csv(f"{AOU_PCA_PREFIX}.eigenvec", sep=r"\s+")
reproj = pd.read_csv(f"{AOU_REPROJECT_PREFIX}.sscore", sep=r"\s+")

direct_id_col = "#IID" if "#IID" in direct.columns else "IID"
reproj_id_col = "#IID" if "#IID" in reproj.columns else "IID"
merged = direct.merge(reproj, left_on=direct_id_col, right_on=reproj_id_col, suffixes=("_direct", "_reproj"))

score_cols = [c for c in reproj.columns if c.startswith("PC") and c.endswith("_AVG")]
for i, pc in enumerate(range(1, 21)):
    direct_col = f"PC{pc}"
    score_col = score_cols[i] if i < len(score_cols) else None
    if direct_col in merged.columns and score_col in merged.columns:
        corr = merged[direct_col].corr(merged[score_col])
        print(f"PC{pc}: r = {corr:.4f}")

## Plot: reference EUR populations projected into AoU's own PC space

Same population palette/marker convention as `round1_filter.ipynb`/`round2_filter.ipynb`/`pc_biplots.ipynb`. Only the two `--score`-derived layers are plotted -- the round-1-passing AoU samples' reprojected scores (background, grey) and the 1000G EUR reference's projected scores (foreground, full alpha) -- not the AoU-fit PCA's own direct `.eigenvec` output, so both layers are on the same `--score`/AVG-scale footing.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

SUPERPOP_POPS = {
    "EUR": ["CEU", "TSI", "FIN", "GBR", "IBS"],
    "AFR": ["YRI", "LWK", "GWD", "MSL", "ESN", "ASW", "ACB"],
    "EAS": ["CHB", "JPT", "CHS", "CDX", "KHV"],
    "SAS": ["GIH", "PJL", "BEB", "STU", "ITU"],
    "AMR": ["MXL", "PUR", "CLM", "PEL"],
}
SUPERPOP_CMAPS = {"EUR": "Blues", "AFR": "Oranges", "EAS": "Greens", "SAS": "Purples", "AMR": "Reds"}
SUPERPOP_SHADE_RANGES = {
    "EUR": (0.25, 0.95), "AFR": (0.35, 0.90), "EAS": (0.30, 0.95), "SAS": (0.25, 0.85), "AMR": (0.35, 0.90),
}
POP_MARKERS_CYCLE = ["o", "s", "^", "D", "v", "P", "X", "*"]


def build_pop_colors(superpop_pops=SUPERPOP_POPS, superpop_cmaps=SUPERPOP_CMAPS, shade_ranges=SUPERPOP_SHADE_RANGES):
    colors = {}
    for superpop, pops in superpop_pops.items():
        cmap = matplotlib.colormaps[superpop_cmaps[superpop]]
        lo, hi = shade_ranges[superpop]
        shades = np.linspace(lo, hi, len(pops)) if len(pops) > 1 else [(lo + hi) / 2]
        for pop, frac in zip(pops, shades):
            colors[pop] = mcolors.to_hex(cmap(frac))
    return colors


def build_pop_markers(superpop_pops=SUPERPOP_POPS, markers_cycle=POP_MARKERS_CYCLE):
    markers = {}
    for pops in superpop_pops.values():
        for i, pop in enumerate(pops):
            markers[pop] = markers_cycle[i % len(markers_cycle)]
    return markers


POP_COLORS = build_pop_colors()
POP_MARKERS = build_pop_markers()

aou_reproj = pd.read_csv(f"{AOU_REPROJECT_PREFIX}.sscore", sep=r"\s+")
aou_reproj_id_col = "#IID" if "#IID" in aou_reproj.columns else "IID"
aou_reproj = aou_reproj.rename(columns={aou_reproj_id_col: "sample"})

ref_proj = pd.read_csv(f"{REF_PROJECT_PREFIX}.sscore", sep=r"\s+")
ref_id_col = "#IID" if "#IID" in ref_proj.columns else "IID"
ref_proj = ref_proj.rename(columns={ref_id_col: "sample"})
ref_proj = ref_proj.merge(panel[["sample", "pop", "super_pop"]], on="sample")

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(
    aou_reproj["PC1_AVG"], aou_reproj["PC2_AVG"],
    c="grey", marker="x", alpha=0.3, s=10, label="AoU (round-1-passing, reprojected)",
)
for pop, grp in ref_proj.groupby("pop"):
    ax.scatter(
        grp["PC1_AVG"], grp["PC2_AVG"],
        c=POP_COLORS.get(pop, "black"), marker=POP_MARKERS.get(pop, "o"),
        label=pop, alpha=1.0, s=15,
    )
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("1000G EUR reference projected into AoU's own PC space")
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "reverse_pca_projection.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Loadings diagnostics: scree plot + Manhattan plot

Same method as `pca_loadings_r2.ipynb`, applied to this AoU-fit PCA -- checks whether the leading PCs carry real signal (scree plot) and whether any PC is dominated by a handful of SNPs in one genomic region rather than genome-wide structure (Manhattan plot + top-N table).

In [ ]:
eigenval = pd.read_csv(f"{AOU_PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title("Reverse PCA (AoU-fit) scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "reverse_pca_scree.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

In [ ]:
weights_path = f"{AOU_PCA_PREFIX}.eigenvec.allele"
weights = pd.read_csv(weights_path, sep="\t")

id_col = "ID"
a1_col = "A1"
pc_cols_raw = [f"PC{i}" for i in range(1, N_PCS_TO_PLOT + 1)]
assert id_col in weights.columns and a1_col in weights.columns, f"unexpected columns: {list(weights.columns)}"
assert all(c in weights.columns for c in pc_cols_raw), f"missing PC columns, have: {list(weights.columns)}"

id_parts = weights[id_col].str.split(":", expand=True)
weights["chr"] = id_parts[0]
weights["pos"] = id_parts[1].astype(int)
weights["alt"] = id_parts[3]

loadings = weights[weights[a1_col] == weights["alt"]].copy()
print(f"{len(weights)} total rows -> {len(loadings)} one-row-per-variant (ALT-allele) loadings")


def chr_sort_key(c):
    c = c.replace("chr", "")
    return int(c) if c.isdigit() else 100


chr_order = sorted(loadings["chr"].unique(), key=chr_sort_key)
chr_offset = {}
offset = 0
for c in chr_order:
    chr_offset[c] = offset
    offset += loadings.loc[loadings["chr"] == c, "pos"].max() + 1
loadings["plot_x"] = loadings.apply(lambda r: chr_offset[r["chr"]] + r["pos"], axis=1)

chr_colors = {c: ("#4a4a4a" if i % 2 == 0 else "#a0a0a0") for i, c in enumerate(chr_order)}

for pc in pc_cols_raw:
    fig, ax = plt.subplots(figsize=(12, 3))
    for c in chr_order:
        sub = loadings[loadings["chr"] == c]
        ax.scatter(sub["plot_x"], sub[pc], s=4, color=chr_colors[c], alpha=0.6, rasterized=True)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_ylabel(f"{pc} loading")
    ax.set_xlabel("Genomic position (chromosomes concatenated)")
    ax.set_title(f"Reverse PCA (AoU-fit) {pc} loadings by position")
    plt.tight_layout()
    plot_path = os.path.join(OUT_DIR, f"reverse_pca_loadings_{pc}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {plot_path}")

    top = loadings.reindex(loadings[pc].abs().sort_values(ascending=False).index).head(N_TOP_LOADINGS)
    print(f"\nTop {N_TOP_LOADINGS} |{pc}| loadings:")
    print(top[["ID", "chr", "pos", pc]].to_string(index=False))
    print()